# Shikibu 学習・書き出し（Colab L4 / レイヤースタック `k3k3` 対応）

`halfkahm2_512x2-16-32_f_k3k3`（9 組レイヤースタック）および `halfkahm2_512x2-16-32_f`（標準 `nn.bin`）を **Colab L4 GPU** で学習し、量子化済み重みを書き出すためのノートブックです。

- **使い方**：メニュー「ランタイム」→「ランタイムのタイプを変更」で **L4 GPU** を選び、「ランタイム」→「すべてのセルを実行」を実行してください。
- 学習は裏のプロセスで走り、`mon.watch()` が進捗を表示し続けます（途中で切れても再実行で `checkpoint.pt` から `--resume` します）。

## 1. Google Drive をマウントする

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 設定（そのまま実行できます。必要に応じて変更してください）

In [ ]:
REPO_URL = "https://github.com/Harumichi-Toshiki/Shikibu.git"
BRANCH = "feat/k3k3-colab-notebook"
REPO = "/content/Shikibu"

# Google Drive 上のパス
WHEEL_DIR = "/content/drive/MyDrive/DainagonProject/匠生用/wheels"
DATA_GLOB = "/content/drive/MyDrive/DainagonProject/匠生用/データセット/psv/*.psv"
VAL_PATH = ""  # 検証用 PSV を分ける場合はここに 1 ファイル指定
OUT_ROOT = "/content/drive/MyDrive/DainagonProject/匠生用/出力"
RUN_NAME = "halfkahm2_512x2-16-32_f_k3k3"

# 構造の指定
#   - 9組レイヤースタック: "halfkahm2_512x2-16-32_f_k3k3"
#   - やねうら王標準 nn.bin: "halfkahm2_512x2-16-32_f"
ARCH = "halfkahm2_512x2-16-32_f_k3k3"

# 学習ハイパーパラメータ（L4 向け・M9 疎計算有効）
CACHE_DIR = "/content/shk_cache"
BATCH_SIZE = 65536
LR = 1e-3
TOTAL_STEPS = 179000
MAX_STEPS = 179000
LAM = 0.75
SAVE_EVERY_MINUTES = 15.0
VAL_EVERY_STEPS = 1000 if VAL_PATH else 0
SEED = 0

## 3. Shikibu リポジトリを取得・更新する

In [ ]:
import os, subprocess

os.makedirs(WHEEL_DIR, exist_ok=True)
if os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "-C", REPO, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "pull", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
print("Repository ready:", REPO, "branch:", BRANCH)

## 4. `shikibu_feat`（Rust 拡張）のインストールと GPU 確認

In [ ]:
import glob, shutil, sys

wheels = sorted(glob.glob(os.path.join(WHEEL_DIR, "shk_feat-*.whl")), key=os.path.getmtime)
if wheels:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", wheels[-1]], check=True)
    print("保存済み wheel をインストール:", wheels[-1])
else:
    if shutil.which("cargo") is None:
        subprocess.run("curl -sSf https://sh.rustup.rs | sh -s -- -y", shell=True, check=True)
    env = dict(os.environ, PATH=os.path.expanduser("~/.cargo/bin") + os.pathsep + os.environ["PATH"])
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "maturin"], check=True)
    subprocess.run(["maturin", "build", "--release", "-m", "crates/shk_feat/Cargo.toml", "-o", WHEEL_DIR],
                   cwd=REPO, env=env, check=True)
    built = sorted(glob.glob(os.path.join(WHEEL_DIR, "shk_feat-*.whl")), key=os.path.getmtime)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", built[-1]], check=True)
    print("wheel をビルドして保存:", built[-1])

import torch, shikibu_feat
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "なし")
print("CPU cores:", os.cpu_count(), "| shikibu_feat OK")

## 5. データセットの確認

In [ ]:
psv_files = sorted(glob.glob(DATA_GLOB))
print(f"検出された PSV ファイル数: {len(psv_files)}")
if not psv_files:
    raise FileNotFoundError(f"PSV が見つかりません: {DATA_GLOB}")
for p in psv_files[:10]:
    print(" ", os.path.basename(p), f"({os.path.getsize(p) / (1024**2):.1f} MB)")
if len(psv_files) > 10:
    print(f"  ... 他 {len(psv_files) - 10} ファイル")

## 6. 学習設定（`config.toml`）を生成する

In [ ]:
sys.path.insert(0, os.path.join(REPO, "python"))
from shikibu import lab, networks

OUT_DIR = os.path.join(OUT_ROOT, RUN_NAME)
os.makedirs(OUT_DIR, exist_ok=True)
CONFIG = os.path.join(OUT_DIR, "config.toml")

cfg = lab.make_config(
    ARCH,
    train_glob=DATA_GLOB,
    out_dir=OUT_DIR,
    val_path=VAL_PATH,
    overrides={
        "data.cache_dir": CACHE_DIR,
        "data.max_cached": 3,
        "data.shuffle_buffer": 20000000,
        "data.num_threads": 11,
        "train.batch_size": BATCH_SIZE,
        "train.lr": LR,
        "train.weight_decay": 1e-4,
        "train.warmup_steps": 1000,
        "train.total_steps": TOTAL_STEPS,
        "train.max_steps": MAX_STEPS,
        "train.min_lr_ratio": 0.01,
        "train.lam": LAM,
        "train.eval_scale": "auto",
        "train.scale_sample": 10000000,
        "train.val_every_steps": VAL_EVERY_STEPS,
        "train.save_every_minutes": SAVE_EVERY_MINUTES,
        "train.seed": SEED,
        "train.ft_forward": "sparse",
        "train.ft_grad": "sparse",
        "train.ft_optimizer": "lazy_adam",
    },
)
lab.write_config(cfg, CONFIG)
print("設定を書き出しました:", CONFIG)
print("構造情報:", networks.parse_arch(ARCH))

## 7. 学習を裏で開始し、進捗を監視する（`mon.watch`）

In [ ]:
sys.path.insert(0, os.path.join(REPO, "tools"))
import shk_monitor as mon

ckpt = os.path.join(OUT_DIR, "checkpoint.pt")
resume = os.path.exists(ckpt)
print("再開モード (--resume):", resume)
mon.start(CONFIG, repo=REPO)
mon.watch(CONFIG)

## 8. 学習済みモデルを書き出す（`k3k3` 9組バイナリ / `nn.bin`）

In [ ]:
parsed = networks.parse_arch(ARCH)
if parsed["stacks"] == 9:
    out_bin = os.path.join(OUT_DIR, f"{parsed['canonical']}.bin")
    cmd = [sys.executable, "-m", "shikibu.export", "--checkpoint", ckpt, "--out", out_bin, "--k3k3"]
else:
    out_bin = os.path.join(OUT_DIR, "nn.bin")
    cmd = [sys.executable, "-m", "shikibu.export", "--checkpoint", ckpt, "--out", out_bin]

env = dict(os.environ, PYTHONPATH=os.path.join(REPO, "python"))
subprocess.run(cmd, cwd=REPO, env=env, check=True)